# 3D Reconstruction Pre-processing Pipeline## **Goal**: Convert raw DICOM CT volumes into paired training data for the# Hybrid ConvNeXt + Swin Transformer U-Net architecture.## **Outputs per case**:# - Binary voxel grid (128³) — ground truth 3D bone structure# - Synthetic 2D X-ray projections (256×256) via dual DRR pipeline## **DRR Dual Pipeline**:# - *Healthy CTs* → MONAI AffineTransform + ray-casting (sum projection)# - *Fractured CTs* → DeepDRR physics-based simulation (fallback to MONAI if unavailable)

In [ ]:
import os
import sys
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.transform import resize as sk_resize
from tqdm.auto import tqdm

# MONAI
import torch
from monai.transforms import (
    Affine,
    EnsureChannelFirst,
)

# DeepDRR availability check
try:
    import deepdrr
    from deepdrr import geo, Volume
    from deepdrr.projector import Projector
    DEEPDRR_AVAILABLE = True
    print("✓ DeepDRR is available — will use for fractured CT DRR generation")
except ImportError:
    DEEPDRR_AVAILABLE = False
    print("✗ DeepDRR not installed — fractured CTs will fall back to MONAI-based DRR")

# ── Project paths (from testproject.config) ──
PROJ_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJ_ROOT))
from testproject.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, INTERIM_DATA_DIR

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════
# PIPELINE CONFIGURATION — edit these as needed
# ══════════════════════════════════════════════
TARGET_VOXEL_GRID_SIZE = (128, 128, 128)   # 3D ground truth resolution
TARGET_DRR_SIZE        = (256, 256)         # 2D X-ray projection resolution
TARGET_SPACING         = (1.0, 1.0, 1.0)   # Isotropic 1 mm resampling
BONE_HU_THRESHOLD      = 300               # HU threshold for bone binarization
HU_WINDOW              = (-1000, 2000)      # HU clamp range
ORIENTATION            = "RAS"             # Standard radiological orientation
MIN_SLICES             = 2                  # Minimum Z-slices to keep a case

# Derived: normalised bone threshold for [0, 1] scaled volumes
BONE_HU_THRESHOLD_NORM = (BONE_HU_THRESHOLD - HU_WINDOW[0]) / (HU_WINDOW[1] - HU_WINDOW[0])
print(f"Normalised bone threshold: {BONE_HU_THRESHOLD_NORM:.4f}")

# DRR projection angles (rotation around Y, X in degrees)
DRR_ANGLES = {
    "ap":      (0, 0),       # Anterior-Posterior
    "lateral": (90, 0),      # Lateral
    "oblique": (45, 0),      # Oblique 45°
}

# ── Case labels: set each case to "healthy" or "fractured" ──
# Fractured cases use DeepDRR; healthy cases use MONAI-based DRR.
# UPDATE THIS DICT with your ground-truth labels.
CASE_LABELS = {
    # PartLeft
    "Case1": "healthy", "Case2": "healthy", "Case3": "healthy",
    # Case4 excluded (single slice)
    # PartRight
    "Case5": "fractured", "Case6": "fractured", "Case7": "fractured",
    "Case8": "fractured", "Case9": "fractured",
    # Case10 excluded (single slice)
    "Case11": "fractured", "Case12": "fractured", "Case13": "fractured",
    "Case14": "fractured", "Case15": "fractured", "Case16": "fractured",
}

# Output directories
VOXEL_DIR = PROCESSED_DATA_DIR / "voxels"
XRAY_DIR  = PROCESSED_DATA_DIR / "xrays"
VOXEL_DIR.mkdir(parents=True, exist_ok=True)
XRAY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJ_ROOT}")
print(f"Raw data     : {RAW_DATA_DIR}")
print(f"Processed out: {PROCESSED_DATA_DIR}")

## 1. Data Discovery & Filtering
Scan raw data directories, build a case manifest, and filter out unusable cases.

In [ ]:
def discover_cases(base_dir, part_name):
    """Find all case directories under a part directory."""
    cases = []
    if base_dir.exists():
        for case_path in sorted(base_dir.iterdir()):
            if case_path.is_dir():
                cases.append({
                    "case_id": case_path.name,
                    "part": part_name,
                    "case_path": str(case_path),
                })
    return cases

# Discover all cases
all_cases = []
all_cases.extend(discover_cases(RAW_DATA_DIR / "PartLeft",  "PartLeft"))
all_cases.extend(discover_cases(RAW_DATA_DIR / "PartRight", "PartRight"))
df_cases = pd.DataFrame(all_cases)

# Count slices per case for filtering
def count_slices(case_path):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(case_path)
    return len(dicom_names)

df_cases["num_slices"] = df_cases["case_path"].apply(count_slices)

# Filter out single-slice cases
excluded = df_cases[df_cases["num_slices"] < MIN_SLICES]
df_valid = df_cases[df_cases["num_slices"] >= MIN_SLICES].reset_index(drop=True)

print(f"Total cases found : {len(df_cases)}")
print(f"Excluded (< {MIN_SLICES} slices): {len(excluded)}")
if len(excluded) > 0:
    print(f"  → Excluded IDs: {excluded['case_id'].tolist()}")
print(f"Valid cases       : {len(df_valid)}")
display(df_valid[["case_id", "part", "num_slices"]])

## 2. DICOM Loading & Metadata Extraction
Load each valid case as a SimpleITK image and collect volume properties.

In [ ]:
def load_dicom_volume(case_path):
    """Load a DICOM series from a directory and return a SimpleITK Image."""
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(case_path)
    reader.SetFileNames(dicom_names)
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    image = reader.Execute()
    return image

# Load metadata for all valid cases
metadata_rows = []
for _, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Loading metadata"):
    img = load_dicom_volume(row["case_path"])
    arr = sitk.GetArrayFromImage(img)
    metadata_rows.append({
        "case_id":   row["case_id"],
        "part":      row["part"],
        "size":      img.GetSize(),
        "spacing":   tuple(round(s, 4) for s in img.GetSpacing()),
        "origin":    tuple(round(o, 2) for o in img.GetOrigin()),
        "hu_min":    int(arr.min()),
        "hu_max":    int(arr.max()),
        "direction": img.GetDirection(),
    })

df_meta = pd.DataFrame(metadata_rows)
display(df_meta[["case_id", "part", "size", "spacing", "hu_min", "hu_max"]])

## 3. Spatial Standardization

Standardize the geometry and coordinate system of each CT volume so that
all cases share the same **orientation**, **voxel spacing**, and **grid size**.

| Step | Operation | Detail |
|------|-----------|--------|
| 3.1 | **Orientation** | Reorient to canonical RAS (Right-Anterior-Superior) axes via `sitk.DICOMOrient` |
| 3.2 | **Isotropic Resampling** | Resample to 1.0 × 1.0 × 1.0 mm using B-spline interpolation; air-fill (−1000 HU) for out-of-bounds |
| 3.3 | **Foreground Cropping** | Bounding-box crop on voxels > −900 HU to discard empty air margins |
| 3.4 | **Resize to Target Grid** | Resize to `TARGET_VOXEL_GRID_SIZE` (128³) with bilinear interpolation, preserving the HU value range |

In [ ]:
def spatial_standardize(sitk_image):
    """
    Spatial standardization pipeline for a single CT volume.

    Handles all geometry / coordinate operations:
      1. Orientation → RAS
      2. Isotropic resampling to TARGET_SPACING
      3. Foreground cropping (remove air margins)
      4. Resize to TARGET_VOXEL_GRID_SIZE

    Returns
    -------
    volume : np.ndarray, float32, shape TARGET_VOXEL_GRID_SIZE
        Spatially standardized volume **still in Hounsfield Units**.
    original_spacing : tuple
        The voxel spacing of the input image (before resampling).
    """
    # ── 3.1  Orientation → RAS ──
    oriented = sitk.DICOMOrient(sitk_image, ORIENTATION)

    # ── 3.2  Isotropic resampling to TARGET_SPACING ──
    original_spacing = oriented.GetSpacing()
    original_size = oriented.GetSize()
    new_size = [
        int(round(osz * ospc / nspc))
        for osz, ospc, nspc in zip(original_size, original_spacing, TARGET_SPACING)
    ]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(TARGET_SPACING)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(oriented.GetDirection())
    resampler.SetOutputOrigin(oriented.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetInterpolator(sitk.sitkBSpline)
    resampler.SetDefaultPixelValue(-1000)          # air fill
    resampled = resampler.Execute(oriented)

    # ── 3.3  Convert to numpy (Z, Y, X) ──
    volume = sitk.GetArrayFromImage(resampled).astype(np.float32)

    # ── 3.4  Foreground cropping ──
    mask = volume > -900
    coords = np.argwhere(mask)
    if coords.size > 0:
        z0, y0, x0 = coords.min(axis=0)
        z1, y1, x1 = coords.max(axis=0) + 1
        volume = volume[z0:z1, y0:y1, x0:x1]

    # ── 3.5  Resize to TARGET_VOXEL_GRID_SIZE ──
    volume = sk_resize(
        volume,
        TARGET_VOXEL_GRID_SIZE,
        order=1,            # bilinear
        mode="constant",
        cval=-1000,
        anti_aliasing=True,
        preserve_range=True,
    ).astype(np.float32)

    return volume, original_spacing

print("✓ spatial_standardize() defined")

In [ ]:
# ── Spatial standardization diagnostic ──
test_img = load_dicom_volume(df_valid.iloc[0]["case_path"])
test_case_id = df_valid.iloc[0]["case_id"]

test_vol_hu, test_spacing = spatial_standardize(test_img)

print(f"Case            : {test_case_id}")
print(f"Original spacing: {tuple(round(s, 4) for s in test_img.GetSpacing())} mm")
print(f"Target spacing  : {TARGET_SPACING} mm")
print(f"Output shape    : {test_vol_hu.shape}  (target: {TARGET_VOXEL_GRID_SIZE})")
print(f"HU range        : [{test_vol_hu.min():.0f}, {test_vol_hu.max():.0f}]")

# Quick visual check — axial / coronal / sagittal mid-slices
d, h, w = test_vol_hu.shape
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(test_vol_hu[d // 2], cmap="gray"); axes[0].set_title("Axial (mid-Z)")
axes[1].imshow(test_vol_hu[:, h // 2, :], cmap="gray"); axes[1].set_title("Coronal (mid-Y)")
axes[2].imshow(test_vol_hu[:, :, w // 2], cmap="gray"); axes[2].set_title("Sagittal (mid-X)")
for ax in axes: ax.axis("off")
fig.suptitle(f"Spatial Standardization — {test_case_id}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Radiometric Standardization

Standardize the **intensity / Hounsfield-Unit values** so that all cases
share the same dynamic range. This is applied *after* spatial
standardization so that interpolation artefacts from resampling are
handled before intensity mapping.

| Step | Operation | Detail |
|------|-----------|--------|
| 4.1 | **HU Windowing** | Clamp values to `HU_WINDOW` = (−1000, 2000) — removes scanner-specific outliers |
| 4.2 | **Min-Max Normalization** | Linear scaling of clamped HU to \[0, 1\]: `(v − HU_min) / (HU_max − HU_min)` |

> **Note**: The bone-binarization threshold is automatically converted to
> the normalised scale (`BONE_HU_THRESHOLD_NORM`) to stay consistent.

In [ ]:
def radiometric_standardize(volume):
    """
    Radiometric standardization for a spatially-standardized CT volume.

    Handles all intensity / grey-value operations:
      1. HU windowing  — clamp to HU_WINDOW
      2. Min-max normalisation — scale to [0, 1]

    Parameters
    ----------
    volume : np.ndarray, float32
        Spatially standardized volume in Hounsfield Units.

    Returns
    -------
    volume_norm : np.ndarray, float32
        Volume with values in [0, 1].
    """
    # ── 4.1  HU windowing ──
    volume = np.clip(volume, HU_WINDOW[0], HU_WINDOW[1])

    # ── 4.2  Min-max normalisation to [0, 1] ──
    hu_min, hu_max = float(HU_WINDOW[0]), float(HU_WINDOW[1])
    volume_norm = (volume - hu_min) / (hu_max - hu_min)

    return volume_norm.astype(np.float32)


def preprocess_ct_volume(sitk_image):
    """
    Convenience wrapper: run the full spatial + radiometric pipeline.

    Returns
    -------
    volume_norm : np.ndarray   — normalised [0, 1] volume
    volume_hu   : np.ndarray   — HU-scale volume (for DRR generation)
    original_spacing : tuple
    """
    volume_hu, original_spacing = spatial_standardize(sitk_image)
    volume_norm = radiometric_standardize(volume_hu)
    return volume_norm, volume_hu, original_spacing

print("✓ radiometric_standardize() defined")
print("✓ preprocess_ct_volume() wrapper defined")

In [ ]:
# ── Radiometric standardization diagnostic ──
test_vol = radiometric_standardize(test_vol_hu)

print(f"Before radiometric — HU range : [{test_vol_hu.min():.0f}, {test_vol_hu.max():.0f}]")
print(f"After  radiometric — range    : [{test_vol.min():.4f}, {test_vol.max():.4f}]")
print(f"Expected threshold (norm)     : {BONE_HU_THRESHOLD_NORM:.4f}")

d, h, w = test_vol.shape
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Normalised slices
axes[0].imshow(test_vol[d // 2], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Normalised Axial"); axes[0].axis("off")

axes[1].imshow(test_vol[:, h // 2, :], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Normalised Coronal"); axes[1].axis("off")

axes[2].imshow(test_vol[:, :, w // 2], cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Normalised Sagittal"); axes[2].axis("off")

# Histogram with threshold
axes[3].hist(test_vol.flatten(), bins=100, color="steelblue", alpha=0.7, log=True)
axes[3].axvline(BONE_HU_THRESHOLD_NORM, color="red", linestyle="--",
                label=f"Bone threshold = {BONE_HU_THRESHOLD_NORM:.4f}")
axes[3].set_title("Normalised Intensity Distribution")
axes[3].set_xlabel("Normalised value [0, 1]")
axes[3].legend(fontsize=8)

fig.suptitle(f"Radiometric Standardization — {test_case_id}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Binary Voxel Grid Generation (Ground Truth)
Threshold the **normalised** volume to produce a binary bone mask.
The threshold is automatically converted from raw HU to the [0, 1] scale
via `BONE_HU_THRESHOLD_NORM`. Morphological closing fills small holes.

In [ ]:
def generate_binary_voxel_grid(volume_norm, threshold=None):
    """
    Binarize a normalised [0, 1] volume into bone (1) vs non-bone (0).
    Applies morphological closing to fill small holes.

    Parameters
    ----------
    volume_norm : np.ndarray
        Normalised volume with values in [0, 1].
    threshold : float, optional
        Normalised threshold. Defaults to BONE_HU_THRESHOLD_NORM.
    """
    if threshold is None:
        threshold = BONE_HU_THRESHOLD_NORM

    binary = (volume_norm >= threshold).astype(np.uint8)

    # Morphological closing to fill small gaps in bone
    struct = ndimage.generate_binary_structure(3, 1)
    binary = ndimage.binary_closing(binary, structure=struct, iterations=2).astype(np.uint8)

    return binary

# Test on first case (using normalised volume)
test_voxels = generate_binary_voxel_grid(test_vol)
bone_ratio = test_voxels.sum() / test_voxels.size * 100
print(f"Voxel grid shape : {test_voxels.shape}")
print(f"Bone voxel ratio : {bone_ratio:.2f}%")
print(f"Dtype            : {test_voxels.dtype}")
print(f"Threshold used   : {BONE_HU_THRESHOLD_NORM:.4f} (≡ {BONE_HU_THRESHOLD} HU)")

## 6. Dual DRR Pipeline — Synthetic X-ray Generation

Two backends for Digitally Reconstructed Radiograph (DRR) generation:

| Pipeline | For | Method |
|---|---|---|
| **MONAI-based** | Healthy CTs | `monai.transforms.Affine` rotation + Beer-Lambert sum projection |
| **DeepDRR** | Fractured CTs | Physics-based simulation (scatter, noise, detector) |

Cases are routed based on `CASE_LABELS` in the configuration cell.

> **Note**: The DRR pipeline operates on the **HU-scale** volume (before
> radiometric normalisation) because it needs physical attenuation values.

In [ ]:
def _rotate_volume(volume, angle_y_deg, angle_x_deg):
    """Rotate a 3D volume using scipy affine for DRR projection angles."""
    angle_y = np.deg2rad(angle_y_deg)
    angle_x = np.deg2rad(angle_x_deg)

    # Rotation matrix: Ry * Rx
    cy, sy = np.cos(angle_y), np.sin(angle_y)
    cx, sx = np.cos(angle_x), np.sin(angle_x)

    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    R = Ry @ Rx

    # Rotate around volume center
    center = np.array(volume.shape) / 2.0
    offset = center - R @ center

    rotated = ndimage.affine_transform(
        volume, R, offset=offset, order=1, mode="constant", cval=-1000
    )
    return rotated


def generate_drr_monai(volume, target_size=TARGET_DRR_SIZE, angles=None):
    """
    Generate DRRs via MONAI-compatible rotation + Beer-Lambert sum projection.

    Parameters
    ----------
    volume : np.ndarray (Z, Y, X) — spatially standardized HU volume
    target_size : tuple — output 2D resolution
    angles : dict — {name: (angle_y, angle_x)} in degrees

    Returns
    -------
    dict of {view_name: np.ndarray} — normalized DRR images
    """
    if angles is None:
        angles = DRR_ANGLES

    # Shift HU to linear attenuation approximation (μ ∝ HU + 1000)
    mu_volume = np.clip(volume + 1000, 0, None)

    drrs = {}
    for name, (ay, ax) in angles.items():
        # Rotate volume to viewing angle
        if ay == 0 and ax == 0:
            rotated = mu_volume
        else:
            rotated = _rotate_volume(mu_volume, ay, ax)

        # Sum projection along Z axis (Beer-Lambert: I = I0 * exp(-sum(mu*dx)))
        projection = np.sum(rotated, axis=0).astype(np.float32)

        # Normalize to [0, 1]
        pmin, pmax = projection.min(), projection.max()
        if pmax > pmin:
            projection = (projection - pmin) / (pmax - pmin)
        else:
            projection = np.zeros_like(projection)

        # Invert (X-ray convention: bone = bright)
        projection = 1.0 - projection

        # Resize to target
        projection = sk_resize(
            projection, target_size, order=1, mode="constant",
            anti_aliasing=True, preserve_range=True
        ).astype(np.float32)

        drrs[name] = projection

    return drrs

print("✓ MONAI-based DRR generator defined")

In [ ]:
def generate_drr_deepdrr(volume, spacing, target_size=TARGET_DRR_SIZE, angles=None):
    """
    Generate DRRs using DeepDRR physics-based simulation.
    Falls back to MONAI-based DRR if DeepDRR is not available.

    Parameters
    ----------
    volume : np.ndarray (Z, Y, X) — spatially standardized HU volume
    spacing : tuple — original voxel spacing in mm
    target_size : tuple — output 2D resolution
    angles : dict — {name: (angle_y, angle_x)} in degrees

    Returns
    -------
    dict of {view_name: np.ndarray} — DRR images
    """
    if not DEEPDRR_AVAILABLE:
        print("  ⚠ DeepDRR unavailable, falling back to MONAI-based DRR")
        return generate_drr_monai(volume, target_size, angles)

    if angles is None:
        angles = DRR_ANGLES

    try:
        # Create DeepDRR volume from numpy array
        # DeepDRR expects (Z, Y, X) array + spacing in mm
        vol = Volume.from_hu(
            volume,
            materials=deepdrr.default_materials,
            anatomical_coordinate_system="RAS",
            world_from_anatomical=geo.FrameTransform.identity(3),
        )

        drrs = {}
        for name, (ay, ax) in angles.items():
            # Set up C-arm at specified angles
            carm = deepdrr.MobileCArm(
                isocenter=vol.center_in_world,
                alpha=float(ay),
                beta=float(ax),
            )

            with Projector(vol, carm=carm, step=0.1,
                           photon_count=10000, scatter_num=0) as projector:
                image = projector()

            # Convert to numpy and normalize
            image = np.array(image).astype(np.float32)
            imin, imax = image.min(), image.max()
            if imax > imin:
                image = (image - imin) / (imax - imin)

            # Resize
            image = sk_resize(
                image, target_size, order=1, mode="constant",
                anti_aliasing=True, preserve_range=True
            ).astype(np.float32)

            drrs[name] = image

        return drrs

    except Exception as e:
        print(f"  ⚠ DeepDRR failed ({e}), falling back to MONAI-based DRR")
        return generate_drr_monai(volume, target_size, angles)

print("✓ DeepDRR generator defined (available: {})".format(DEEPDRR_AVAILABLE))

In [ ]:
def generate_drr(case_id, volume_hu, spacing, target_size=TARGET_DRR_SIZE):
    """
    Route DRR generation through the correct pipeline based on CASE_LABELS.
    Healthy → MONAI,  Fractured → DeepDRR (with fallback).

    Parameters
    ----------
    volume_hu : np.ndarray — spatially standardized volume in HU (NOT normalised)
    """
    label = CASE_LABELS.get(case_id, "healthy")

    if label == "fractured":
        print(f"  [{case_id}] Using DeepDRR pipeline (fractured)")
        return generate_drr_deepdrr(volume_hu, spacing, target_size)
    else:
        print(f"  [{case_id}] Using MONAI pipeline (healthy)")
        return generate_drr_monai(volume_hu, target_size)

# Test DRR on first case — uses the HU-scale volume
test_drrs = generate_drr(test_case_id, test_vol_hu, test_spacing)
print(f"\nGenerated {len(test_drrs)} DRR views: {list(test_drrs.keys())}")
for name, drr in test_drrs.items():
    print(f"  {name}: shape={drr.shape}, range=[{drr.min():.3f}, {drr.max():.3f}]")

## 7. Visualization & Quality Check
Display the full pipeline results for a sample case: CT mid-slices → binary voxel grid → DRR projections.

In [ ]:
def visualize_pipeline(case_id, volume_hu, voxel_grid, drrs):
    """
    Comprehensive visualization of pipeline outputs for one case.

    Parameters
    ----------
    volume_hu : np.ndarray — HU-scale spatially standardized volume (for display)
    """
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle(f"Pre-processing Pipeline — {case_id}", fontsize=16, fontweight="bold")

    d, h, w = volume_hu.shape
    mid_z, mid_y, mid_x = d // 2, h // 2, w // 2

    # Row 1: Preprocessed CT mid-slices (HU)
    ax1 = fig.add_subplot(3, 4, 1)
    ax1.imshow(volume_hu[mid_z], cmap="gray", aspect="auto")
    ax1.set_title("CT Axial (mid-Z)")
    ax1.axis("off")

    ax2 = fig.add_subplot(3, 4, 2)
    ax2.imshow(volume_hu[:, mid_y, :], cmap="gray", aspect="auto")
    ax2.set_title("CT Coronal (mid-Y)")
    ax2.axis("off")

    ax3 = fig.add_subplot(3, 4, 3)
    ax3.imshow(volume_hu[:, :, mid_x], cmap="gray", aspect="auto")
    ax3.set_title("CT Sagittal (mid-X)")
    ax3.axis("off")

    # HU histogram
    ax4 = fig.add_subplot(3, 4, 4)
    ax4.hist(volume_hu.flatten(), bins=100, color="steelblue", alpha=0.7, log=True)
    ax4.axvline(BONE_HU_THRESHOLD, color="red", linestyle="--", label=f"Threshold={BONE_HU_THRESHOLD}")
    ax4.set_title("HU Distribution")
    ax4.legend(fontsize=8)

    # Row 2: Binary voxel grid mid-slices
    ax5 = fig.add_subplot(3, 4, 5)
    ax5.imshow(voxel_grid[mid_z], cmap="bone", aspect="auto")
    ax5.set_title("Voxel Grid Axial")
    ax5.axis("off")

    ax6 = fig.add_subplot(3, 4, 6)
    ax6.imshow(voxel_grid[:, mid_y, :], cmap="bone", aspect="auto")
    ax6.set_title("Voxel Grid Coronal")
    ax6.axis("off")

    ax7 = fig.add_subplot(3, 4, 7)
    ax7.imshow(voxel_grid[:, :, mid_x], cmap="bone", aspect="auto")
    ax7.set_title("Voxel Grid Sagittal")
    ax7.axis("off")

    # Bone density info
    ax8 = fig.add_subplot(3, 4, 8)
    bone_pct = voxel_grid.sum() / voxel_grid.size * 100
    ax8.text(0.5, 0.5, f"Bone Voxels\n{voxel_grid.sum():,}\n({bone_pct:.1f}%)",
             ha="center", va="center", fontsize=14, transform=ax8.transAxes,
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightcoral", alpha=0.5))
    ax8.set_title("Bone Statistics")
    ax8.axis("off")

    # Row 3: DRR projections
    drr_names = list(drrs.keys())
    for i, name in enumerate(drr_names[:4]):
        ax = fig.add_subplot(3, 4, 9 + i)
        ax.imshow(drrs[name], cmap="gray", aspect="auto")
        ax.set_title(f"DRR — {name.upper()}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# Visualize first case — pass the HU volume for display
visualize_pipeline(test_case_id, test_vol_hu, test_voxels, test_drrs)

## 8. Batch Processing Pipeline
Process all valid cases through the full pipeline and save outputs to `data/processed/`.

In [ ]:
manifest = []

for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Processing cases"):
    case_id   = row["case_id"]
    case_path = row["case_path"]
    part      = row["part"]

    try:
        # 1. Load DICOM
        sitk_image = load_dicom_volume(case_path)
        original_size    = sitk_image.GetSize()
        original_spacing = sitk_image.GetSpacing()

        # 2. Spatial standardization (returns HU-scale volume)
        volume_hu, spacing = spatial_standardize(sitk_image)

        # 3. Radiometric standardization (returns [0, 1] volume)
        volume_norm = radiometric_standardize(volume_hu)

        # 4. Generate binary voxel grid from normalised volume
        voxel_grid = generate_binary_voxel_grid(volume_norm)

        # 5. Generate DRRs from HU volume (needs physical attenuation values)
        drrs = generate_drr(case_id, volume_hu, spacing)

        # 6. Save voxel grid
        voxel_path = VOXEL_DIR / f"{case_id}.npy"
        np.save(str(voxel_path), voxel_grid)

        # 7. Save DRR projections
        drr_paths = {}
        for view_name, drr_img in drrs.items():
            drr_path = XRAY_DIR / f"{case_id}_{view_name}.npy"
            np.save(str(drr_path), drr_img)
            drr_paths[view_name] = str(drr_path)

        # 8. Record manifest entry
        bone_ratio = float(voxel_grid.sum() / voxel_grid.size)
        manifest.append({
            "case_id":          case_id,
            "part":             part,
            "label":            CASE_LABELS.get(case_id, "healthy"),
            "original_size":    list(original_size),
            "original_spacing": [round(s, 4) for s in original_spacing],
            "voxel_path":       str(voxel_path),
            "drr_paths":        drr_paths,
            "bone_ratio":       round(bone_ratio, 4),
            "drr_pipeline":     "deepdrr" if CASE_LABELS.get(case_id) == "fractured" and DEEPDRR_AVAILABLE else "monai",
        })

        print(f"  ✓ {case_id} — bone ratio: {bone_ratio*100:.1f}%")

    except Exception as e:
        print(f"  ✗ {case_id} — ERROR: {e}")
        manifest.append({"case_id": case_id, "part": part, "error": str(e)})

# Save manifest
manifest_path = PROCESSED_DATA_DIR / "processing_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\n✓ Manifest saved to {manifest_path}")
print(f"  Successfully processed: {sum(1 for m in manifest if 'error' not in m)}/{len(manifest)}")

## 9. Train / Validation / Test Split
Split processed cases into training, validation, and test sets. Stratify by `part` when possible.

In [ ]:
from sklearn.model_selection import train_test_split

# Filter successful cases
successful = [m for m in manifest if "error" not in m]
case_ids   = [m["case_id"] for m in successful]
parts      = [m["part"] for m in successful]

# Split: ~70/15/15  (with small dataset, do integer splits)
n = len(case_ids)
n_test = max(1, round(n * 0.15))
n_val  = max(1, round(n * 0.15))

# First split off test
train_val_ids, test_ids = train_test_split(
    case_ids, test_size=n_test, random_state=42,
    stratify=parts if len(set(parts)) > 1 and n_test >= len(set(parts)) else None,
)

# Then split train/val
parts_tv = [parts[case_ids.index(c)] for c in train_val_ids]
train_ids, val_ids = train_test_split(
    train_val_ids, test_size=n_val, random_state=42,
    stratify=parts_tv if len(set(parts_tv)) > 1 and n_val >= len(set(parts_tv)) else None,
)

splits = {"train": sorted(train_ids), "val": sorted(val_ids), "test": sorted(test_ids)}
splits_path = PROCESSED_DATA_DIR / "splits.json"
with open(splits_path, "w") as f:
    json.dump(splits, f, indent=2)

print("Data Split Summary:")
print(f"  Train : {len(train_ids)} cases — {sorted(train_ids)}")
print(f"  Val   : {len(val_ids)} cases  — {sorted(val_ids)}")
print(f"  Test  : {len(test_ids)} cases  — {sorted(test_ids)}")
print(f"\n✓ Splits saved to {splits_path}")

## 10. Pipeline Summary
Final statistics and verification of the processed dataset.

In [ ]:
print("=" * 60)
print("PRE-PROCESSING PIPELINE SUMMARY")
print("=" * 60)

successful = [m for m in manifest if "error" not in m]
failed     = [m for m in manifest if "error" in m]

print(f"\nCases processed  : {len(successful)}/{len(df_valid)}")
if failed:
    print(f"Cases failed     : {len(failed)} — {[m['case_id'] for m in failed]}")

bone_ratios = [m["bone_ratio"] for m in successful]
print(f"\nBone voxel ratio:")
print(f"  Mean  : {np.mean(bone_ratios)*100:.2f}%")
print(f"  Min   : {np.min(bone_ratios)*100:.2f}%")
print(f"  Max   : {np.max(bone_ratios)*100:.2f}%")

pipelines_used = [m["drr_pipeline"] for m in successful]
print(f"\nDRR Pipeline usage:")
print(f"  MONAI-based  : {pipelines_used.count('monai')}")
print(f"  DeepDRR      : {pipelines_used.count('deepdrr')}")

print(f"\nOutput files:")
voxel_files = list(VOXEL_DIR.glob("*.npy"))
xray_files  = list(XRAY_DIR.glob("*.npy"))
print(f"  Voxel grids (.npy) : {len(voxel_files)} in {VOXEL_DIR}")
print(f"  X-ray DRRs (.npy)  : {len(xray_files)} in {XRAY_DIR}")
print(f"  Manifest           : {manifest_path}")
print(f"  Splits             : {splits_path}")

# Verify shapes
print(f"\nShape verification (random sample):")
if voxel_files:
    sample_voxel = np.load(str(voxel_files[0]))
    print(f"  Voxel: {sample_voxel.shape}, dtype={sample_voxel.dtype}")
if xray_files:
    sample_xray = np.load(str(xray_files[0]))
    print(f"  X-ray: {sample_xray.shape}, dtype={sample_xray.dtype}")

print("\n" + "=" * 60)
print("✓ Pipeline complete!")
print("=" * 60)